In [19]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle

import pandas as pd
import numpy as np

In [33]:
model = load_model('model.h5') ##  ANN



In [34]:
## load the encoder and scaler
with open('One_hot_encoder.pkl','rb') as file:
    label_encoder_geo=pickle.load(file)

with open('label_encoder.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('Scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [35]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [36]:
# One-hot encode 'Geography'
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df


C:\Users\victus\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [37]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [38]:
## Encode categorical variables
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

In [39]:
input_df['Gender']

0    1
Name: Gender, dtype: int64

In [40]:
## concatination one hot encoded 
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)


In [41]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [42]:
order = ['Geography_France','Geography_Germany'	,'Geography_Spain',	'CreditScore',	'Gender',	'Age',	'Tenure',	'Balance',	'NumOfProducts',	'HasCrCard',	'IsActiveMember',	'EstimatedSalary']
input_df = input_df[order]
input_df

,Geography_France,Geography_Germany,Geography_Spain,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,1.0,0.0,0.0,600,1,40,3,60000,2,1,1,50000


In [43]:
## Scaling the input data
input_scaled = scaler.transform(input_df)
input_scaled

array([[ 1.00150113, -0.57946723, -0.57638802, -0.53598516,  0.91324755,
         0.10479359, -0.69539349, -0.25781119,  0.80843615,  0.64920267,
         0.97481699, -0.87683221]])

In [44]:
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step


array([[0.02912563]], dtype=float32)

In [45]:
prob = prediction[0][0]

In [46]:
if prob> 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
